In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_49.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_90.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_77.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_66.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_54.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_42.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_81.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_72.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final

In [2]:
import pandas as pd 
BASE="/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

train=pd.read_csv(BASE +"/train.csv")
test=pd.read_csv(BASE +"/test.csv")
sample=pd.read_csv(BASE +"/sample_submission.csv")

print("Train:")
print(train.head())
print("\nTrain shape:",train.shape)

print("Test:")
print(test.head())
print("\nTest shape:",test.shape)

print("\nSAMPLE SUBMISSION:")
print(sample.head())

Train:
        filename  label
0  audio_192.wav    3.0
1  audio_436.wav    3.0
2  audio_447.wav    3.5
3  audio_126.wav    2.0
4   audio_61.wav    2.0

Train shape: (769, 2)
Test:
        filename  label
0  audio_128.wav     -1
1  audio_156.wav     -1
2   audio_19.wav     -1
3  audio_108.wav     -1
4  audio_206.wav     -1

Test shape: (216, 2)

SAMPLE SUBMISSION:
         filename  label
0   audio_804.wav     -1
1  audio_1028.wav     -1
2   audio_865.wav     -1
3   audio_774.wav     -1
4  audio_1138.wav     -1


In [3]:
import os
import numpy as np
import pandas as pd
import librosa

TRAIN_AUDIO = BASE + "/train"
TEST_AUDIO = BASE + "/test"

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)

    features = []

    # MFCC features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    # Basic audio features
    features.append(np.mean(librosa.feature.zero_crossing_rate(y)))
    features.append(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    features.append(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)))
    features.append(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
    features.append(np.mean(librosa.feature.rms(y=y)))

    # Duration
    features.append(len(y) / sr)

    return np.array(features)


def build_features(df, audio_folder):
    X = []

    for i, filename in enumerate(df["filename"]):
        path = os.path.join(audio_folder, filename)
        X.append(extract_features(path))

        if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{len(df)}")

    return np.array(X)


print("Extracting TRAIN features...")
X_train = build_features(train, TRAIN_AUDIO)

print("Extracting TEST features...")
X_test = build_features(test, TEST_AUDIO)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Extracting TRAIN features...
Processed 50/769
Processed 100/769
Processed 150/769
Processed 200/769
Processed 250/769
Processed 300/769
Processed 350/769
Processed 400/769
Processed 450/769
Processed 500/769
Processed 550/769
Processed 600/769
Processed 650/769
Processed 700/769
Processed 750/769
Extracting TEST features...
Processed 50/216
Processed 100/216
Processed 150/216
Processed 200/216
X_train shape: (769, 46)
X_test shape: (216, 46)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import numpy as np

y = train["label"].values

# Training/validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y, test_size=0.2, random_state=42
)

# Model
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

print("Training model...")
model.fit(X_tr, y_tr)

# Validation prediction
val_pred = model.predict(X_val)

# Metrics
rmse = np.sqrt(mean_squared_error(y_val, val_pred))
pearson = pearsonr(y_val, val_pred)[0]

print("Validation RMSE:", rmse)
print("Validation Pearson Correlation:", pearson)

Training model...
Validation RMSE: 0.7377453008174816
Validation Pearson Correlation: 0.8394258577739776


In [5]:
from sklearn.ensemble import ExtraTreesRegressor

model2 = ExtraTreesRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

print("Training improved model...")
model2.fit(X_tr, y_tr)

val_pred2 = model2.predict(X_val)

rmse2 = np.sqrt(mean_squared_error(y_val, val_pred2))
pearson2 = pearsonr(y_val, val_pred2)[0]

print("Improved Validation RMSE:", rmse2)
print("Improved Validation Pearson:", pearson2)

Training improved model...
Improved Validation RMSE: 0.6927162048343574
Improved Validation Pearson: 0.861480910745904


In [6]:
# Final model: train on all available training data

final_model = ExtraTreesRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

print("Training final model on all 769 training samples...")
final_model.fit(X_train, y)

# Mandatory training-data RMSE
train_pred = final_model.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y, train_pred))

print("Training RMSE:", train_rmse)

# Predict test data
print("Generating test predictions...")
test_pred = final_model.predict(X_test)

# Grammar score should be between 1 and 5
test_pred = np.clip(test_pred, 1, 5)

# Create submission in the required format
submission = pd.DataFrame({
    "filename": test["filename"],
    "label": test_pred
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("\nSubmission created successfully!")
print(submission.head())
print("Submission shape:", submission.shape)

Training final model on all 769 training samples...
Training RMSE: 0.10422143579670205
Generating test predictions...

Submission created successfully!
        filename     label
0  audio_128.wav  3.306833
1  audio_156.wav  3.845500
2   audio_19.wav  3.454167
3  audio_108.wav  2.442667
4  audio_206.wav  3.198833
Submission shape: (216, 2)
